# 3. Cost breakdowns and sensitivity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/03_cost_breakdowns_and_sensitivity.ipynb)

Totals hide where the money goes. This notebook uses OpenPyTEA's analysis and plotting functions to break costs
down, draw the cumulative cash-flow diagram, and ask *"what if"* questions with one-way sensitivity curves and
tornado plots.

## Learning goals

- Use the `*_data()` + `plot_*()` pattern behind every OpenPyTEA plot.
- Plot cost breakdowns for one or several plants, and read a cumulative cash-flow diagram.
- Run one-way sensitivity analyses and find break-even points.
- Rank cost drivers with a tornado plot and know its limitations.

## 0) Setup: rebuild the ammonia plant from notebooks 01 and 02

In [ ]:
# Run this cell. It installs OpenPyTEA if it is missing (e.g., on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
from openpytea import Equipment

# Equipment of the green-ammonia loop (from notebook 01)
compressor  = Equipment(name="C-101 Syngas compressor", param=1_200,
                        process_type="Fluids",
                        category="Compressors, fans, & blowers",
                        type="Compressor, centrifugal", material="Carbon steel")
motor       = Equipment(name="M-101 Compressor motor", param=1_200 / 0.90,
                        process_type="Electrical",
                        category="Motors & generators",
                        type="Totally enclosed motor")
exchanger   = Equipment(name="E-101 Feed/effluent exchanger", param=250,
                        process_type="Fluids",
                        category="Heat exchangers", type="U-tube shell & tube",
                        material="304 stainless steel")
converter   = Equipment(name="R-101 Ammonia converter", param=12_000,
                        process_type="Fluids",
                        category="Reactors", type="Tubular fixed bed",
                        material="304 stainless steel")
separator   = Equipment(name="V-101 Ammonia separator", param=25,
                        process_type="Fluids",
                        category="Pressure vessels", type="Vertical",
                        material="Carbon steel")
chiller     = Equipment(name="K-101 Refrigeration unit", param=800,
                        process_type="Fluids",
                        category="Utilities",
                        type="Packaged mechanical refrigerator")
product_pump = Equipment(name="P-101 Product pump", param=20,
                         process_type="Fluids",
                         category="Pumps", type="Single-stage centrifugal",
                         material="Carbon steel")

equipment_list = [compressor, motor, exchanger, converter, separator, chiller,
                  product_pump]

In [ ]:
from openpytea import Plant

ammonia_config = {
    "plant_name": "Green Ammonia",
    "process_type": "Fluids",
    "country": "Netherlands",
    "equipment": equipment_list,

    # Financial assumptions
    "interest_rate": 0.08,
    "project_lifetime": 20,      # years
    "plant_utilization": 0.90,   # fraction of the year on stream
    "tax_rate": 0.25,

    # Products: the first entry is the main product used for the levelized cost
    "plant_products": {
        "ammonia": {"production": 100_000, "price": 0.95},   # kg/day, USD/kg
    },

    # Variable OPEX: daily consumption and unit price
    "variable_opex_inputs": {
        # kg/day,  USD/kg
        "hydrogen":    {"consumption": 17_800, "price": 2.00},
        # kg/day,  USD/kg
        "nitrogen":    {"consumption": 82_400, "price": 0.05},
        # kWh/day, USD/kWh
        "electricity": {"consumption": 60_000, "price": 0.08},
    },
}

ammonia_plant = Plant(ammonia_config)
ammonia_plant.calculate_all()
print(f"Fixed capital investment : {ammonia_plant.fixed_capital:,.0f} USD")
print(f"Levelized cost of ammonia: {ammonia_plant.levelized_cost:.3f} USD/kg")
print(f"Net present value        : {ammonia_plant.npv:,.0f} USD")

## 1) The data + plot pattern

Every visualization in OpenPyTEA is two functions: a `*_data()` function in `openpytea.analysis` that returns a
plain dictionary, and a `plot_*()` function in `openpytea.plotting` that draws it with matplotlib. You can
therefore reuse the numbers in a report or in your own plotting code.

| Data function | Plot function | Shows |
|--------------------------|------------------------------------|--------------------------------------|
| `direct_costs_data` | `plot_stacked_bar` | Direct cost per equipment item |
| `fixed_capital_data` | `plot_stacked_bar` | ISBL, OSBL, D&E, contingency (and additional CAPEX) |
| `variable_opex_data` | `plot_stacked_bar` | Variable OPEX per feed and utility |
| `fixed_opex_data` | `plot_stacked_bar` | Fixed OPEX components |
| `levelized_cost_data` | `plot_stacked_bar` | LCOP split into discounted CAPEX, OPEX, and side revenue |
| `cash_flow_data` | `plot_cash_flow` | Cumulative cash flow over the project life |
| `sensitivity_data` | `plot_sensitivity` | One-way sensitivity curve |
| `tornado_data` | `plot_tornado` | One-at-a-time ranking of parameters |
| `monte_carlo` | `plot_monte_carlo`, `plot_multiple_monte_carlo`, `plot_monte_carlo_inputs` | Notebook 04 |

In [ ]:
from openpytea.analysis import *
from openpytea.plotting import *
import matplotlib.pyplot as plt

## 2) Cost breakdowns

In [ ]:
# equipment-level CAPEX
fig, ax = plot_stacked_bar(direct_costs_data(ammonia_plant))
# FCI components
fig, ax = plot_stacked_bar(fixed_capital_data(ammonia_plant))

In [ ]:
# feeds and utilities
fig, ax = plot_stacked_bar(variable_opex_data(ammonia_plant))
# fixed OPEX in %
fig, ax = plot_stacked_bar(fixed_opex_data(ammonia_plant, pct=True))

In [ ]:
# Where does the levelized cost come from? Discounted CAPEX vs. OPEX
# (by-product credits would appear as negative bars)
lcop_breakdown = levelized_cost_data(ammonia_plant)
fig, ax = plot_stacked_bar(lcop_breakdown)

Passing a **list of plants** puts one bar per plant side by side. Let's compare the base case with the cheaper-hydrogen variant from notebook 02.

In [ ]:
from copy import deepcopy

cheap_h2 = deepcopy(ammonia_plant)
cheap_h2.update_configuration({
    "plant_name": "Cheaper H_2",
    "variable_opex_inputs": {"hydrogen": {"price": 1.50}},
})
cheap_h2.calculate_all()

fig, ax = plot_stacked_bar(levelized_cost_data([ammonia_plant, cheap_h2]))
fig, ax = plot_stacked_bar(variable_opex_data([ammonia_plant, cheap_h2]))

## 3) Cumulative cash-flow diagram

The diagram shows the dip into debt during construction, the point of maximum investment, the break-even year,
and the final cumulative position. The returned dictionary carries these numbers so you can quote them directly.

In [ ]:
cash_flow = cash_flow_data([ammonia_plant, cheap_h2])
fig, ax = plot_cash_flow(cash_flow)

for curve in cash_flow["curves"]:
    breakeven = curve["breakeven_year"]
    breakeven_text = (f"break-even in Year {breakeven:.1f}"
                      if breakeven is not None else "no break-even")
    print(f"{curve['plant']} - Max investment: "
          f"{curve['max_investment']:,.0f} USD, "
          f"in Year {curve['max_investment_year']:.0f}; {breakeven_text}")

## 4) One-way sensitivity curves

`sensitivity_data()` varies **one** parameter by ± a fraction of its base value and returns the curve of a
`metric` ( e.g., `LCOP`, `ROI`, or `PBT`). The parameter can be a price (`"hydrogen"`, `"ammonia"`), a
financial assumption (`"interest_rate"`, `"fixed_capital"`), or a process quantity (`"hydrogen.consumption"`,
`"ammonia.production"`).

In [ ]:
h2_sensitivity = sensitivity_data(
    ammonia_plant, parameter="hydrogen", plus_minus_value=0.5, metric="LCOP",
    label="Levelized cost of ammonia / [USD/kg]")
fig, ax = plot_sensitivity(h2_sensitivity)

In [ ]:
# Several plants on one figure, and a profitability metric this time
npv_vs_price = sensitivity_data([ammonia_plant, cheap_h2], parameter="ammonia",
                                plus_minus_value=0.3, metric="NPV", n_points=31)
fig, ax = plot_sensitivity(npv_vs_price)

The returned dictionary has one entry per plant under `"curves"`, each with the relative change `x` and
the metric `y`. With `numpy.interp` we can read the break-even point:

In [ ]:
import numpy as np

curve = npv_vs_price["curves"][0]                  # base-case plant
pct = np.interp(0.0, curve["y"], curve["x"])       # NPV rises with the price
price = ammonia_plant.plant_products["ammonia"]["price"] * (1 + pct / 100)
print(f"NPV = 0 at {pct:+.1f} % of the base price, i.e. {price:.3f} USD/kg")

## 5) Tornado plots

A tornado plot varies every parameter one at a time by the same ± fraction and ranks them by the swing they
cause, showing which inputs deserve better data. `category` selects the parameters: `"economic"` (prices and
financial assumptions, the default), `"process"` (consumption and production), or `"both"`.

In [ ]:
lcop_tornado = tornado_data(ammonia_plant, plus_minus_value=0.3, metric="LCOP")
fig, ax = plot_tornado(lcop_tornado)

In [ ]:
npv_tornado = tornado_data(ammonia_plant, plus_minus_value=0.3, metric="NPV",
                           category="process")
fig, ax = plot_tornado(npv_tornado)

> **A caveat.** Varying `Ammonia prod.` alone while the feed consumptions stay fixed is not physically
> possible: less ammonia means less hydrogen consumed. Notebook 04 shows how to link such parameters so that the sweep respects them.

## 6) Task 3.1: the cost of capital

Compare the base plant (`ammonia_plant`) and the cheaper-hydrogen plant (`cheap_h2`) from section 2: pass both
as a list to `sensitivity_data`.

1. Vary the **interest rate** by ± 50 % (from 4 to 12 %) and plot two metrics, the **LCOP** and the **IRR**.
   Which metric responds to interest rate, and why does the other one not?
2. Vary the **ammonia price** by ± 30 % and plot the **IRR**. Use `numpy.interp` to find, for each plant, the
   ammonia price at which the IRR equals the 8 % interest rate (the IRR rises with the price, so no flipping is
   needed). Compare it with the NPV = 0 break-even price of section 4. What do you notice?

In [ ]:
# YOUR CODE HERE

## 7) Task 3.2 (bonus): depreciation and working capital

Make two variants of `ammonia_plant` and draw their cumulative cash flows together with the base case in one
`plot_cash_flow` diagram:

1. MACRS (Modified Accelerated Cost Recovery System) depreciation instead of the default straight-line
   depreciation: `"depreciation": {"method": "macrs"}`.
2. A working capital of 25 % of the FCI instead of 15 %: `"fixed_opex_factors": {"working_capital": 0.25}`.

How do the maximum investment and the break-even year change? Which of the two variants changes the LCOP, and
why?

For details on depreciation methods, see the [Cash flow](https://openpytea.readthedocs.io/en/stable/user_guide/plant.html#cash-flow) section of the `Plant` user guide or [part 2 of the OpenPyTEA walkthrough](https://github.com/pbtamarona/OpenPyTEA/blob/main/walkthrough/part_2_plant.ipynb).

In [ ]:
# YOUR CODE HERE

## 8) Task 3.3 (bonus): make it publication-ready

Every plot function accepts `show=False` and returns `(fig, ax)`, so you can restyle it with matplotlib. Pick any
plot from this notebook, style it the way you like, and save it with `fig.savefig(...)`.

For examples, see the [Analysis user guide](https://openpytea.readthedocs.io/en/stable/user_guide/analysis.html) or [part 3 of the OpenPyTEA walkthrough](https://github.com/pbtamarona/OpenPyTEA/blob/main/walkthrough/part_3_analysis.ipynb).

In [ ]:
# YOUR CODE HERE

## Recap

- `*_data()` returns numbers, `plot_*()` draws them; pass a list of plants to compare.
- Sensitivity curves find break-even points; tornado plots rank the cost drivers.
- Both vary one input at a time; the next notebook varies them all at once.

**Next:** [4. Monte Carlo uncertainty analysis](04_monte_carlo_uncertainty.ipynb)